In [ ]:
# This example will demonstrate:

    # Defining a Custom Tool: How to create a Python class that encapsulates the logic for interacting with your MCP server.
    # Creating Agents: How to define AI agents with specific roles, goals, and backstories.
    # Defining Tasks: How to set up tasks that your agents will perform, utilizing the custom tools.
    # Orchestrating a Crew: How to bring agents, MCP and tasks together into a collaborative crew.

# --- 0. Import CrewAI & MCPServerAdapter ---

In [ ]:
from crewai import Agent
from crewai_tools import MCPServerAdapter
from crewai import Task, Crew, LLM
from crewai import Agent, Task, Crew, Process, LLM

In [ ]:
# 2. SSE Server:
# # MCP configuration memory MCP servers
server_params = [
    {
    "url": "http://localhost:8000/sse",
    "transport": "sse"
    }
  ]

with MCPServerAdapter(server_params) as mcp_tools:
    print(f"Available tools: {[tool.name for tool in mcp_tools]}")

# This snippet sets up a connection to an MCP (Multi-Agent Control Protocol) Server using Server-Sent Events (SSE) 
# to dynamically load tools that agents can use in a CrewAI workflow.

# Agent: Represents an autonomous CrewAI agent that can perform tasks using tools.
# MCPServerAdapter: A connector that allows CrewAI to fetch tools from an external MCP server.

# url: Points to the local MCP server endpoint that streams tool metadata via SSE.
# transport: Specifies the protocol (sse) used for real-time communication.

# with MCPServerAdapter(...): Initializes the connection to the MCP server and automatically cleans up afterward.
# with MCPServerAdapter(server_params) as mcp_tools:: This is a with statement, a Python construct that ensures resources are properly managed.
# In this case, it handles the connection to the MCP server.
# mcp_tools: A list-like object containing tool instances fetched from the server.
# tool.name: Prints the name of each tool available to the agent.


# --- 1. Import Opik ---

In [ ]:
# Import opik and its CrewAI integration
import opik
from opik.integrations.crewai import track_crewai
track_crewai(project_name="arunmanglick-crewai-integration-demo")

# --- 1. Setup Local LLM ---

In [ ]:
# The LLM class is used to configure a language model.
# We are specifying 'ollama/llama3.2' as the model and pointing to the default local Ollama server address.

local_llm = LLM(
    model="ollama/llama3.2",
    base_url="http://localhost:11434"
)

# What is Ollama
# Ollama is a platform that lets you run LLMs locally on your machine—no cloud dependency required. 
# It’s designed for developers who want fast, private, and customizable access to models like LLaMA, Mistral, Gemma, Phi-4, and more.


# --- 2. Define AI Agent ---

In [ ]:
# from crewai import LLM

# # Uncomment and configure your local LLM (Ollama) if not already done
local_llm = LLM(
    model="ollama/llama3.2",
    base_url="http://localhost:11434"
)

# This code to use local LLM is commented this time, as the code is using the LLM from OpenAI
# To use OpenAI, logged-in here https://platform.openai.com/ 
# Generated Key 
# Added the 'OPENAI_API_KEY' in env file
# ------------------------------------------------------------

with MCPServerAdapter(server_params) as mcp_tools:
    print(f"Available tools: {[tool.name for tool in mcp_tools]}")

    # Memory Agent
    memory_agent = Agent(
        role="Memory Reset",
        goal="Clear Memory information",
        backstory="You are an agent to clean up memory information.",
        allow_delegation=False,
        tools=[mcp_tools["clear_graph"]],
        llm=local_llm  # Assign the local LLM to this agent
    )

    
    memory_task = Task(
        description="Store the query in memory: {query}.",
        agent=memory_agent,
        expected_output="Confirmation that the information has been stored.",
    )

    crew = Crew(
        agents=[memory_agent],
        tasks=[memory_task],
        verbose=True
    )

    result = crew.kickoff(inputs={})
    print(result)